In [ ]:
import os
from pathlib import Path

import pandas as pd


In [ ]:
path = Path.cwd()
#path
#data_path = f"{path.parent}/data"
data_path = path.parent / "data"
data_path

In [ ]:
# load the dataset
df_raw = pd.read_csv(data_path / "csic_2010_raw.csv", low_memory=False)
#df_raw.columns.tolist()
df_raw


**Observations:**  

The raw CSV contains:

- The raw dataset contains approximately 223,585 parameter level rows and 18 columns.
- These rows may represent logical HTTP requests grouped by `index`.  
- The `method` represents HTTP methods, such as GET, POST, or PUT. Can be used a categorial column
- The `url` column Request URL or URI. Can be featurised in multiple ways based on:
  - Counts on special chars in url such as '&', '/', '.', '=' etc
  - Count of query parameters
  - Length of `url`
  - Presence of well know suspicious parterns such as:
    - sql: select, union, drop, insert, delete
    - html: script tag
    - Path traversal pattern
- The `protocol` column represents HTTP protocol version. Seems to be a constant value column
- The `user_agent`column represents User-Agent header value. Seems to be a constant value column
- HTTP header field columns `pragma`, `cache_control`, `accept`, `accept_encoding`, `accept_charset`, `accept_language`, `host` and `connection` seem to be constant value columns  
- The `content_length` contains a numeric content length value
- The `contentType`  can be used a categorial column
- The `payload` columns represent the request payload. Can be featurised in multiple ways based on: 
    - Whether payload is present or not
    - Length of `payload`
    - Counts on special chars in payload such as '&', '/', '.', '=' etc
    - Presence of well know suspicious parterns such as:
        - sql: select, union, drop, insert, delete
        - html: script tag
        - Path traversal pattern
- The `label` is the target column with `anom` (anomalous) and `norm` (normal) values

Check the data types and missing values in the various columns

In [ ]:
df_raw.info()

**Observations:**  
- `contentLength`, `contentType`, `payload` colums have values missing.
- The target `label` has no values missing. (i.e no rows to discard)


## 1. Analyze `index` column

**Extract rows for `index=0`**

In [ ]:
group_index0 = df_raw[df_raw['index'] == 0]
group_index0

> **Observations:**  
> - `6` rows in total for `index=0`
> - Rows# `0` to `4` and `119585` implies non-contiguous row distribution for `index=0`

**Find the columns with *constant*, *varying* and *NaN* values**

In [ ]:
const_value_cols = []
varying_value_cols = []
nan_value_cols = []
for col in group_index0.columns:
    # Check if all values in the column are NaN
    if group_index0[col].isna().all():
        nan_value_cols.append(col)
    # Check if the column has exactly one unique value (excluding NaN)
    elif group_index0[col].nunique() == 1:
        const_value_cols.append(col)
    else:
        varying_value_cols.append(col)
print(f"Varying value columns:\n    {varying_value_cols}")
print()
print(f"Constant value columns:\n    {const_value_cols}")
print()
print(f"NaN value columns:\n    {nan_value_cols}")
print()

**Identify Unique Values in Varying Columns:**  

In [ ]:
# Unique URLs
url_unique_values = group_index0['url'].unique()
print(f"Unique URLs:\n{url_unique_values}\n")

# Unique COOKIEs
cookie_unique_values = group_index0['cookie'].unique()
print(f"Unique Cookies:\n{cookie_unique_values}\n")

# Unique LABELs
label_unique_values = group_index0['label'].unique()
print(f"Unique Labels:\n{label_unique_values}\n")

# Unique PAYLOADs
#payload_unique_values = group_index0['payload'].dropna().unique()
payload_unique_values = group_index0['payload'].unique()
#print(f"Unique payloads(excluding NaN):\n{payload_unique_values}\n")
print(f"Unique payloads:\n{payload_unique_values}\n")

> **Observations:**
> - 2 unique URLs: Suggests 2 separate HTTP requests.  
> - 2 unique cookies: Aligns with the 2 URLs, indicating 2 distinct sessions.  
> - 2 unique labels: anom (anomalous) and norm (normal).  
> - 5 unique payloads: Fragments of the same request (for anom) or no payload (for norm)  

**Map Unique Values to index and Row Numbers:**

In [ ]:
# Index list corresponding to two URLs
print("Index values for URLs")

# URL[0]
mask = (group_index0['url'] == url_unique_values[0])
url_0_idx_list = group_index0[group_index0['url'] == url_unique_values[0]]['index']
url_0_row_list = group_index0.loc[mask].index
print(f"  URL[0]: {url_unique_values[0]}")
print(f"    Index: {url_0_idx_list.tolist()}")
print(f"    Rows : {url_0_row_list.tolist()}")

# URL[1]
mask = (group_index0['url'] == url_unique_values[1])
url_1_idx_list = group_index0[group_index0['url'] == url_unique_values[1]]['index']
url_1_row_list = group_index0.loc[mask].index
print(f"  URL[1]: {url_unique_values[1]}")
print(f"    Index: {url_1_idx_list.tolist()}")
print(f"    Rows : {url_1_row_list.tolist()}")
print()
# -------------------------------------------------------------

# Index list corresponding to two COOKIEs
print("Index values for COOKIEs")

# Cookie[0]
mask = (group_index0['cookie'] == cookie_unique_values[0])
cookie_0_idx_list = group_index0[group_index0['cookie'] == cookie_unique_values[0]]['index']
cookie_0_row_list = group_index0.loc[mask].index
print(f"  COOKIE[0]: {cookie_unique_values[0]}")
print(f"    Index: {cookie_0_idx_list.tolist()}")
print(f"    Rows : {cookie_0_row_list.tolist()}")

# Cookie[1]
mask = (group_index0['cookie'] == cookie_unique_values[1])
cookie_1_idx_list = group_index0[group_index0['cookie'] == cookie_unique_values[1]]['index']
cookie_1_row_list = group_index0.loc[mask].index
print(f"  COOKIE[1]: {cookie_unique_values[1]}")
print(f"    Index: {cookie_1_idx_list.tolist()}")
print(f"    Rows : {cookie_1_row_list.tolist()}")
print()
# -------------------------------------------------------------

# Index list corresponding to two LABELs
print("Index values for LABELs")

# LABEL[0]
mask = (group_index0['label'] == label_unique_values[0])
label_0_idx_list = group_index0[group_index0['label'] == label_unique_values[0]]['index']
label_0_row_list = group_index0.loc[mask].index
print(f"  LABEL[0]: {label_unique_values[0]}")
print(f"    Index: {label_0_idx_list.tolist()}")
print(f"    Rows : {label_0_row_list.tolist()}")

# LABEL[1]
mask = (group_index0['label'] == label_unique_values[1])
label_1_idx_list = group_index0[group_index0['label'] == label_unique_values[1]]['index']
label_1_row_list = group_index0.loc[mask].index
print(f"  LABEL[1]: {label_unique_values[1]}")
print(f"    Index: {label_1_idx_list.tolist()}")
print(f"    Rows : {label_1_row_list.tolist()}")
print()
# -------------------------------------------------------------

# Index list corresponding to two PAYLOADs
print("Index values for PAYLOADs")

# PAYLOAD[0]
mask = (group_index0['payload'] == payload_unique_values[0])
payload_0_idx_list = group_index0[group_index0['payload'] == payload_unique_values[0]]['index']
payload_0_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[0]: {payload_unique_values[0]}")
print(f"    Index: {payload_0_idx_list.tolist()}")
print(f"    Rows : {payload_0_row_list.tolist()}")

# PAYLOAD[1]
mask = (group_index0['payload'] == payload_unique_values[1])
payload_1_idx_list = group_index0[group_index0['payload'] == payload_unique_values[1]]['index']
payload_1_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[1]: {payload_unique_values[1]}")
print(f"    Index: {payload_1_idx_list.tolist()}")
print(f"    Rows : {payload_1_row_list.tolist()}")

# PAYLOAD[2]
mask = (group_index0['payload'] == payload_unique_values[2])
payload_2_idx_list = group_index0[group_index0['payload'] == payload_unique_values[2]]['index']
payload_2_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[2]: {payload_unique_values[2]}")
print(f"    Index: {payload_2_idx_list.tolist()}")
print(f"    Rows : {payload_2_row_list.tolist()}")

# PAYLOAD[3]
mask = (group_index0['payload'] == payload_unique_values[3])
payload_3_idx_list = group_index0[group_index0['payload'] == payload_unique_values[3]]['index']
payload_3_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[3]: {payload_unique_values[3]}")
print(f"    Index: {payload_3_idx_list.tolist()}")
print(f"    Rows : {payload_3_row_list.tolist()}")

# PAYLOAD[4]
mask = (group_index0['payload'] == payload_unique_values[4])
#print(mask)
payload_4_idx_list = group_index0[group_index0['payload'] == payload_unique_values[4]]['index']
payload_4_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[4]: {payload_unique_values[4]}")
print(f"    Index: {payload_4_idx_list.tolist()}")
print(f"    Rows : {payload_4_row_list.tolist()}")

# PAYLOAD[5]
#mask = (group_index0['payload'] == payload_unique_values[5])
mask = group_index0['payload'].isna()
#print(mask)
#payload_5_idx_list = group_index0[group_index0['payload'] == payload_unique_values[5]]['index']
payload_5_idx_list = group_index0.loc[mask, 'index']
payload_5_row_list = group_index0.loc[mask].index
print(f"  PAYLOAD[5]: {payload_unique_values[5]}")
print(f"    Index: {payload_5_idx_list.tolist()}")
print(f"    Rows : {payload_5_row_list.tolist()}")
print()

> **Observations:**  
>
> 1. HTTP Request 1: *Derived from unique combination of `URL[0]` + `Cookie[0]`*  
>    - URL: `http://localhost:8080/tienda1/publico/anadir.jsp`  
>    - Cookie: `JSESSIONID=B92A8B48B9008CD29F622A994E0F650D`  
>    - Label: `anom` (anomalous)  
>    - Payload: Fragmented across rows 0–4. Each row contains a part of the payload (e.g., id=2, nombre=Jamón).  
>    - Rows: 0-4  
>    - Index: 0 (reused for all rows in this request).  
> 2. HTTP Request 2: *Derived from unique combination of `URL[1]` + `Cookie[1]`*  
>    - URL: `http://localhost:8080/tienda1/index.jsp`  
>    - Cookie: `JSESSIONID=1F767F17239C9B670A39E9B10C3825F4`  
>    - Label: `norm` (normal)  
>    - Payload: `NaN` i.e no payload, which is normal for GET requests.  
>    - Rows: `119585`
>    - Index: `0` (reused, but distinct from Request 1 due to different url and cookie).
> 3. Index Reuse:  
>    - The `index` column is reused (0 for both requests). However, the combination of `url + cookie` uniquely identifies each request.  